# 00 Run Context And Sources

Este notebook fija el contexto inmutable del run que se va a auditar. Toda auditoria posterior debe reutilizar estas rutas y no redefinir fuentes de verdad localmente.

- Si preguntas: “cuál es el universo histórico amplio del proyecto”
    - mira tickers_all.parquet o tickers_2005_2026.parquet
- Si preguntas: “cuál es la lista operativa de tickers única”
    - mira tickers_2005_2026_upper.parquet
- Si preguntas: “cuál es el universo refinado elegido para quotes de producción”
    - mira tickers_2005_2026_upper_excluding_ohlcv_1m_missing_vs_daily_ge_2B_or_null.parquet
- Si preguntas: “cuál fue el universo final realmente usado en este run”
    - mira tickers_quotes_prod.csv

In [3]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json
import pandas as pd

ENABLE_SHA256 = False
SHA256_SUFFIXES = {".json", ".md", ".csv", ".txt"}

PROJECT_ROOT = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps")
RUN_ID = "20260313_quotes_prod_full_12133_clean"
RUN_DIR = PROJECT_ROOT / "runs" / "polygon_realtime_audit" / RUN_ID
INPUTS_DIR = RUN_DIR / "inputs"
QUOTES_ROOT = Path(r"D:\quotes")

UNIVERSE_REFINED = PROJECT_ROOT / "data" / "reference" / "universe_pti" / "tickers_2005_2026_upper_excluding_ohlcv_1m_missing_vs_daily_ge_2B_or_null.parquet"
OFFICIAL_LIFECYCLE = PROJECT_ROOT / "data" / "reference" / "official_lifecycle_compiled.csv"

TASKS_PROD_CSV = INPUTS_DIR / "tasks_quotes_prod.csv"
TASKS_PROD_META_JSON = INPUTS_DIR / "tasks_quotes_prod_meta.json"
TICKERS_PROD_CSV = INPUTS_DIR / "tickers_quotes_prod.csv"
TASKS_MISSING_FINAL_V2_CSV = INPUTS_DIR / "tasks_quotes_prod_missing_only_final_v2.csv"

DOWNLOAD_EVENTS_CURRENT = RUN_DIR / "download_events_current.csv"
DOWNLOAD_EVENTS_HISTORY = RUN_DIR / "download_events_history.csv"
DOWNLOAD_STATE_JSON = RUN_DIR / "download_state.json"
DOWNLOAD_LIVE_STATUS_JSON = RUN_DIR / "download_live_status.json"

AGENT02_EVENTS_CURRENT = RUN_DIR / "quotes_agent_strict_events_current.csv"
AGENT02_LIVE_STATUS_JSON = RUN_DIR / "live_status_quotes_strict.json"
AGENT02_RETRY_CURRENT_CSV = RUN_DIR / "retry_queue_quotes_strict_current.csv"

AGENT03_DIR = RUN_DIR / "agent03_outputs"
AGENT03_RUN_SUMMARY_JSON = AGENT03_DIR / "run_summary.json"

DISK_INVENTORY_CSV = RUN_DIR / "disk_quotes_inventory.csv"
REPAIRED_CURRENT_CSV = RUN_DIR / "download_events_current.repaired_from_history.csv"
REPAIRED_SUMMARY_JSON = RUN_DIR / "download_events_current.repaired_summary.json"
RECONCILE_SUMMARY_JSON = RUN_DIR / "reconcile_disk_vs_history_summary.json"

def sha256_file(path: Path) -> str | None:
    if not path.exists() or not path.is_file():
        return None
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def should_hash(path: Path) -> bool:
    return ENABLE_SHA256 and path.suffix.lower() in SHA256_SUFFIXES

def describe_path(path: Path, source_role: str) -> dict:
    out = {
        "path": str(path),
        "source_role": source_role,
        "exists": path.exists(),
        "is_file": path.is_file(),
        "is_dir": path.is_dir(),
    }
    if path.exists() and path.is_file():
        st = path.stat()
        out["size_bytes"] = int(st.st_size)
        out["mtime_utc"] = pd.Timestamp(st.st_mtime, unit="s", tz="UTC").isoformat()
        out["sha256"] = sha256_file(path) if should_hash(path) else None
    return out

CONTEXT = {
    "project_root": str(PROJECT_ROOT),
    "run_id": RUN_ID,
    "run_dir": str(RUN_DIR),
    "quotes_root": str(QUOTES_ROOT),
}

SOURCES = {
    "universe_refined": describe_path(UNIVERSE_REFINED, "universe"),
    "official_lifecycle": describe_path(OFFICIAL_LIFECYCLE, "task_builder"),
    "tasks_prod_csv": describe_path(TASKS_PROD_CSV, "task_builder"),
    "tasks_prod_meta_json": describe_path(TASKS_PROD_META_JSON, "task_builder"),
    "tickers_prod_csv": describe_path(TICKERS_PROD_CSV, "task_builder"),
    "tasks_missing_final_v2_csv": describe_path(TASKS_MISSING_FINAL_V2_CSV, "recovery"),
    "download_events_current": describe_path(DOWNLOAD_EVENTS_CURRENT, "agent01"),
    "download_events_history": describe_path(DOWNLOAD_EVENTS_HISTORY, "agent01"),
    "download_state_json": describe_path(DOWNLOAD_STATE_JSON, "agent01"),
    "download_live_status_json": describe_path(DOWNLOAD_LIVE_STATUS_JSON, "agent01"),
    "agent02_events_current": describe_path(AGENT02_EVENTS_CURRENT, "agent02"),
    "agent02_live_status_json": describe_path(AGENT02_LIVE_STATUS_JSON, "agent02"),
    "agent02_retry_current_csv": describe_path(AGENT02_RETRY_CURRENT_CSV, "agent02"),
    "agent03_run_summary_json": describe_path(AGENT03_RUN_SUMMARY_JSON, "agent03"),
    "disk_inventory_csv": describe_path(DISK_INVENTORY_CSV, "recovery"),
    "repaired_current_csv": describe_path(REPAIRED_CURRENT_CSV, "recovery"),
    "repaired_summary_json": describe_path(REPAIRED_SUMMARY_JSON, "recovery"),
    "reconcile_summary_json": describe_path(RECONCILE_SUMMARY_JSON, "recovery"),
}

print(json.dumps(CONTEXT, indent=2))
pd.DataFrame(SOURCES).T

{
  "project_root": "C:\\TSIS_Data\\v1\\backtest_SmallCaps",
  "run_id": "20260313_quotes_prod_full_12133_clean",
  "run_dir": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\runs\\polygon_realtime_audit\\20260313_quotes_prod_full_12133_clean",
  "quotes_root": "D:\\quotes"
}


,path,source_role,exists,is_file,is_dir,size_bytes,mtime_utc,sha256
universe_refined,C:\TSIS_Data\v1\backtest_SmallCaps\data\refere...,universe,True,True,False,81280,2026-03-13T14:25:45.869206190+00:00,None
official_lifecycle,C:\TSIS_Data\v1\backtest_SmallCaps\data\refere...,task_builder,True,True,False,81192,2026-03-13T13:52:40.629931211+00:00,None
tasks_prod_csv,C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygo...,task_builder,True,True,False,51598162,2026-03-14T07:38:40.476658106+00:00,None
tasks_prod_meta_json,C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygo...,task_builder,True,True,False,455,2026-03-14T07:38:40.585658312+00:00,None
tickers_prod_csv,C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygo...,task_builder,True,True,False,102412,2026-03-14T07:38:40.482658148+00:00,None
tasks_missing_final_v2_csv,C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygo...,recovery,True,True,False,15246658,2026-03-18T22:51:23.804486990+00:00,None
download_events_current,C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygo...,agent01,True,True,False,134321244,2026-03-19T12:06:29.369779110+00:00,None
download_events_history,C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygo...,agent01,True,True,False,509202993,2026-03-19T12:06:23.885774374+00:00,None
download_state_json,C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygo...,agent01,True,True,False,501,2026-03-19T12:06:29.470779181+00:00,None
download_live_status_json,C:\TSIS_Data\v1\backtest_SmallCaps\runs\polygo...,agent01,True,True,False,443,2026-03-19T12:06:29.507778882+00:00,None


In [4]:
meta = json.loads(TASKS_PROD_META_JSON.read_text(encoding="utf-8")) if TASKS_PROD_META_JSON.exists() else {}
download_state = json.loads(DOWNLOAD_STATE_JSON.read_text(encoding="utf-8")) if DOWNLOAD_STATE_JSON.exists() else {}
download_live = json.loads(DOWNLOAD_LIVE_STATUS_JSON.read_text(encoding="utf-8")) if DOWNLOAD_LIVE_STATUS_JSON.exists() else {}
agent02_live = json.loads(AGENT02_LIVE_STATUS_JSON.read_text(encoding="utf-8")) if AGENT02_LIVE_STATUS_JSON.exists() else {}
agent03_summary = json.loads(AGENT03_RUN_SUMMARY_JSON.read_text(encoding="utf-8")) if AGENT03_RUN_SUMMARY_JSON.exists() else {}

snapshot = {
    "tasks_meta": meta,
    "download_state": download_state,
    "download_live": download_live,
    "agent02_live": agent02_live,
    "agent03_summary_keys": sorted(agent03_summary.keys()),
}
print(json.dumps(snapshot, indent=2, ensure_ascii=False))

{
  "tasks_meta": {
    "run_id": "20260313_quotes_prod_full_12133_clean",
    "universe_path": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\data\\reference\\universe_pti\\tickers_2005_2026_upper_excluding_ohlcv_1m_missing_vs_daily_ge_2B_or_null.parquet",
    "lifecycle_path": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\data\\reference\\official_lifecycle_compiled.csv",
    "tickers_count": 1961,
    "tasks_total": 3067682,
    "date_min": "1995-09-13",
    "date_max": "2026-03-13"
  },
  "download_state": {
    "run_id": "20260313_quotes_prod_full_12133_clean",
    "updated_utc": "2026-03-19T12:06:29.471409+00:00",
    "tasks_total": 907151,
    "tasks_current_rows": 640781,
    "status_counts_current": {
      "DOWNLOADED_OK": 468936,
      "DOWNLOADED_EMPTY": 171845
    },
    "retry_pending": 0,
    "output_root": "D:\\quotes",
    "csv_path": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\runs\\polygon_realtime_audit\\20260313_quotes_prod_full_12133_clean\\inputs\\tasks_quotes_prod_missing_only_f